In [101]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
import joblib

In [102]:
df = pd.read_csv('insurance.csv')

In [103]:
df.duplicated().sum()

np.int64(0)

In [104]:
df.isna().sum()

Id               0
age              5
gender           0
bmi              0
bloodpressure    0
diabetic         0
children         0
smoker           0
region           3
claim            0
dtype: int64

In [105]:
df.dropna(inplace=True)

In [106]:
df.drop(columns=['Id'], inplace=True)

In [107]:
X = df[['age', 'gender', 'bmi', 'bloodpressure', 'diabetic', 'children', 'smoker']]
y = df[['claim']]

In [108]:
cat_cols = ['gender', 'diabetic', 'smoker']
label_encoders = {}

In [109]:
le = LabelEncoder()
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le
    
    joblib.dump(le, f"label_encoder_{col}.pkl")

C:\Users\noahg\AppData\Local\Temp\ipykernel_11816\1629543755.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(X[col])
C:\Users\noahg\AppData\Local\Temp\ipykernel_11816\1629543755.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(X[col])
C:\Users\noahg\AppData\Local\Temp\ipykernel_11816\1629543755.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

Se

In [110]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.30)

In [111]:
num_cols = ['age', 'bmi', 'bloodpressure', 'children']
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [112]:
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [113]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


In [114]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return {"r2": r2, "MAE": mae, "RMSE": rmse}

In [115]:
results = {}

In [116]:
lr = LinearRegression()
lr.fit(X_train, y_train)

results['Linear Regression'] = evaluate_model(lr, X_train, X_test, y_train, y_test)
print("Linear Regression model trained!")

best_poly_model = None
best_poly_score = -np.inf

for degree in [2,3]:
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    
    poly_lr = LinearRegression()
    poly_lr.fit(X_train_poly, y_train)
    
    score = poly_lr.score(X_test_poly, y_test)
    
    if score > best_poly_score:
        best_poly_score = score
        best_poly_model = (degree, poly, poly_lr)
        
degree, poly, poly_lr = best_poly_model

results[f'Polynomial Regression (deg = {degree})'] = evaluate_model(poly_lr, poly.fit_transform(X_train), poly.transform(X_test), y_train, y_test)
print("Polynomial Regression model trained!")

rf = RandomForestRegressor()

rf_params = {
    'n_estimators' : [100, 200, 500, 1000],
    "max_depth" : [None, 10, 20, 30],
    "min_samples_split" : [2, 5, 10],
    "min_samples_leaf" : [1, 2, 4, 8]
}

rf_grid_obj = GridSearchCV(rf, rf_params, scoring='r2', cv=5, n_jobs=-1, verbose=2, refit=True )
rf_grid_obj.fit(X_train, y_train)
best_rf = rf_grid_obj.best_estimator_
best_rf_params = rf_grid_obj.best_params_

results['Random Forest'] = evaluate_model(best_rf, X_train, X_test, y_train, y_test)
print("Random Forest training is completed!, best parameters:", best_rf_params)

svr = SVR()

svr_params = {
    "kernel" : ['linear', 'rbf', 'poly'],
    "C" : [0.1, 1, 10, 100],
    "gamma" : ['scale', 'auto'],
    "epsilon" : [0.01, 0.1, 0.5],
    "degree" : [2,3]
}

svr_grid_obj = GridSearchCV(svr, svr_params, scoring='r2', n_jobs=-1, cv=5, verbose=2, refit=True)
svr_grid_obj.fit(X_train, y_train)
best_svr = svr_grid_obj.best_estimator_
best_svr_params = svr_grid_obj.best_params_


results['Support Vector Regressor'] = evaluate_model(best_svr, X_train, X_test, y_train, y_test)
print('Support Vector Regressor training is completed!, best parameters is:', best_svr_params)

xgb = XGBRegressor(objective='reg:squarederror')

xgb_params = {
    "n_estimators" : [100, 200, 500, 1000],
    "max_depth" : [3, 5, 7],
    "learning_rate" : [0.01, 0.02, 0.05, 0.1],
    "subsample" : [0.8, 1.0],
    "gamma" : [0.1, 0.5], 
    "reg_alpha": [0, 0.1, 1],            #L1 Regularization
    "reg_lambda": [1, 2, 5]              #L2 Regularization
}

xgb_grid_obj = GridSearchCV(xgb, xgb_params, scoring='r2', cv=5, n_jobs=-1, verbose=2, refit=True)
xgb_grid_obj.fit(X_train, y_train)
best_xgb = xgb_grid_obj.best_estimator_
best_xgb_params = xgb_grid_obj.best_params_

results['XGBoost'] = evaluate_model(best_xgb, X_train, X_test, y_train, y_test)
print('XGBoost training is completed!, best parameters is:', best_xgb_params)


   

Linear Regression model trained!
Polynomial Regression model trained!
Fitting 5 folds for each of 192 candidates, totalling 960 fits


c:\Users\noahg\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Random Forest training is completed!, best parameters: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 200}
Fitting 5 folds for each of 144 candidates, totalling 720 fits


c:\Users\noahg\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Support Vector Regressor training is completed!, best parameters is: {'C': 100, 'degree': 2, 'epsilon': 0.5, 'gamma': 'scale', 'kernel': 'linear'}
Fitting 5 folds for each of 1728 candidates, totalling 8640 fits


c:\Users\noahg\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


XGBoost training is completed!, best parameters is: {'gamma': 0.1, 'learning_rate': 0.02, 'max_depth': 3, 'n_estimators': 200, 'reg_alpha': 0, 'reg_lambda': 1, 'subsample': 1.0}


In [117]:
results

{'Linear Regression': {'r2': 0.7339814824206052,
  'MAE': 4712.873362450279,
  'RMSE': np.float64(6318.004574431195)},
 'Polynomial Regression (deg = 2)': {'r2': 0.8028640242129131,
  'MAE': 4163.471055619484,
  'RMSE': np.float64(5438.848291220653)},
 'Random Forest': {'r2': 0.8238793874023697,
  'MAE': 3918.558137919025,
  'RMSE': np.float64(5140.780906906018)},
 'Support Vector Regressor': {'r2': 0.6808553061004641,
  'MAE': 4809.500176084983,
  'RMSE': np.float64(6920.186766684603)},
 'XGBoost': {'r2': 0.8274644613265991,
  'MAE': 3899.59521484375,
  'RMSE': np.float64(5088.1894618813085)}}

In [118]:
results_df = pd.DataFrame(results).T.sort_values(by='r2', ascending=False)
results_df

,r2,MAE,RMSE
XGBoost,0.827464,3899.595215,5088.189462
Random Forest,0.823879,3918.558138,5140.780907
Polynomial Regression (deg = 2),0.802864,4163.471056,5438.848291
Linear Regression,0.733981,4712.873362,6318.004574
Support Vector Regressor,0.680855,4809.500176,6920.186767


In [119]:
models = {
    'Linear Regression' : lr,
    'Polynomial Regression' : poly_lr,
    'Random Forest' : best_rf,
    'SVR' : best_svr,
    'XGBoost' : best_xgb
}

In [120]:
best_r2 = results_df['r2'].max()
best_r2

np.float64(0.8274644613265991)

In [121]:
top_model = results_df[results_df['r2'] == best_r2]
top_model

,r2,MAE,RMSE
XGBoost,0.827464,3899.595215,5088.189462


In [122]:
best_model = models[top_model.index[0]]
best_model

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [123]:
joblib.dump(best_model, 'best_model.pkl')
print(f"Best Model selected: {top_model.index[0]}")

Best Model selected: XGBoost
